# BƯỚC 1: TRÍCH XUẤT DỮ LIỆU (EXTRACTION)

In [7]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import re
import logging
import traceback
import subprocess
from pathlib import Path

import openpyxl
import pdfplumber
import pytesseract
import torch
from docling.document_converter import DocumentConverter as DoclingConverter
from pdf2image import convert_from_path
from PIL import Image
from transformers import AutoModel, AutoTokenizer
from docx import Document as DocxDocument

logging.basicConfig(level=logging.INFO, format="%(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

SUPPORTED_EXT = {".pdf", ".docx", ".doc", ".xlsx", ".xls", ".pptx", ".ppt", ".md", ".txt"}
PDF_OCR_STYLE = {
    "prompt": "<image>\n<|grounding|>Convert the document to markdown.",
    "base_size": 1024,
    "image_size": 640,
    "crop_mode": True,
    "test_compress": True,
    "save_results": True,
}

In [30]:
from pathlib import Path

# 1. Đường dẫn file PDF đầu vào
pdf_file_path = "/home/trung-ai/chatbot_hcns/doc/cb87654abb87470d810926dcd13fd959_2026.HR.AIPT._Quy_che_Luong_Thuong_Phuc_loi.pdf"

# Khởi tạo đối tượng Path
pdf_path = Path(pdf_file_path)

# 2. Tự động sinh các đường dẫn pipeline
# Đổi đuôi .pdf thành .md (Lưu cùng thư mục doc)
origin_input_file_path = str(pdf_path.with_suffix(".md"))

# Thêm hậu tố _cleaned vào tên file (Lưu cùng thư mục doc)
cleaned_text_file_path = str(pdf_path.with_name(f"{pdf_path.stem}_cleaned.md"))

# --- MỚI THÊM: TỰ ĐỘNG SINH ĐƯỜNG DẪN JSON ---
# pdf_path.parent là thư mục 'doc'
# pdf_path.parent.parent là thư mục 'chatbot_hcns'
project_root_dir = pdf_path.parent.parent 
staging_dir = project_root_dir / "staging_chunks"

# Nối thư mục staging với tên file gốc + hậu tố _chunks.json
json_input_path = str(staging_dir / f"{pdf_path.stem}_cleaned_chunks.json")

# --- In ra để kiểm tra ---
print("⚙️ CẤU HÌNH ĐƯỜNG DẪN PIPELINE:")
print(f"1. PDF Input : {pdf_path}")
print(f"2. Raw MD    : {origin_input_file_path}")
print(f"3. Clean MD  : {cleaned_text_file_path}")
print(f"4. JSON Stage: {json_input_path}")

⚙️ CẤU HÌNH ĐƯỜNG DẪN PIPELINE:
1. PDF Input : /home/trung-ai/chatbot_hcns/doc/cb87654abb87470d810926dcd13fd959_2026.HR.AIPT._Quy_che_Luong_Thuong_Phuc_loi.pdf
2. Raw MD    : /home/trung-ai/chatbot_hcns/doc/cb87654abb87470d810926dcd13fd959_2026.HR.AIPT._Quy_che_Luong_Thuong_Phuc_loi.md
3. Clean MD  : /home/trung-ai/chatbot_hcns/doc/cb87654abb87470d810926dcd13fd959_2026.HR.AIPT._Quy_che_Luong_Thuong_Phuc_loi_cleaned.md
4. JSON Stage: /home/trung-ai/chatbot_hcns/staging_chunks/cb87654abb87470d810926dcd13fd959_2026.HR.AIPT._Quy_che_Luong_Thuong_Phuc_loi_cleaned_chunks.json


In [9]:
from transformers import AutoTokenizer, AutoModel, AutoConfig
import transformers.utils.import_utils
import torch

model_id = "/home/trung-ai/chatbot_hcns/weights/DeepSeek-OCR-bnb-4bit-NF4"

# 1. Vá lỗi Llama (Cũ của bạn)
try:
    from transformers.models.llama import modeling_llama
    if not hasattr(modeling_llama, "LlamaFlashAttention2"):
        setattr(modeling_llama, "LlamaFlashAttention2", modeling_llama.LlamaAttention)
except ImportError:
    pass

# 2. VÁ LỖI CHO DEEPSEEK-OCR TRÊN TRANSFORMERS MỚI (Vá hàm import)
if not hasattr(transformers.utils.import_utils, 'is_torch_fx_available'):
    print("🔧 Đang vá lỗi 'is_torch_fx_available'...")
    transformers.utils.import_utils.is_torch_fx_available = lambda: False

print("⏳ Đang tải Tokenizer và Config...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Tải cấu hình (Config) riêng lẻ ra để xử lý
config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)

# 3. VÁ LỖI THIẾU THUỘC TÍNH (Vá pad_token_id)
if not hasattr(config, 'pad_token_id'):
    print("🔧 Đang vá lỗi thiếu 'pad_token_id' trong Config...")
    # Lấy pad_token từ tokenizer, nếu không có thì lấy eos_token, không có nữa thì gán = 0
    config.pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else getattr(tokenizer, 'eos_token_id', 0)

print("⏳ Đang tải Model OCR lên GPU...")
# Truyền config đã được vá lỗi vào AutoModel
model = AutoModel.from_pretrained(
    model_id,
    config=config, # <-- ĐIỂM MẤU CHỐT NẰM Ở ĐÂY
    trust_remote_code=True,
    use_safetensors=True,
    device_map="auto",
    torch_dtype=torch.bfloat16,
).eval()

print("⏳ Đang tải Docling Converter...")
from docling.document_converter import DocumentConverter
docling_converter = DocumentConverter()

print("✅ Đã sẵn sàng các công cụ chuyển đổi!")

⏳ Đang tải Tokenizer và Config...


[transformers] DeepseekV2ForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
[transformers] DeepseekOCRForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't di

🔧 Đang vá lỗi thiếu 'pad_token_id' trong Config...
⏳ Đang tải Model OCR lên GPU...


Loading weights: 100%|██████████| 2711/2711 [00:02<00:00, 1149.70it/s]
[transformers] DeepseekV2ForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
[transformers] DeepseekOCRForCausalLM has generative capabilities, as `prepa

⏳ Đang tải Docling Converter...
✅ Đã sẵn sàng các công cụ chuyển đổi!


In [10]:
def auto_rotate(image_path: Path) -> None:
    try:
        img = Image.open(image_path)
        osd = pytesseract.image_to_osd(img, output_type=pytesseract.Output.DICT)
        angle = osd.get("rotate", 0)
        if angle:
            img.rotate(-angle, expand=True).save(image_path)
    except Exception:
        pass

def is_pdf_scan(pdf_path: str) -> bool:
    try:
        total_chars = 0
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    total_chars += len(text.strip())
                    if total_chars >= 100:
                        return False
        return True
    except Exception as exc:
        logger.warning("Unable to check PDF scan status: %s", exc)
        return True

def clean_empty_tables(text: str) -> str:
    keywords = {"lần", "số yêu cầu", "sửa đổi", "nội dung", "ngày", "người", "revision", "date", "description", "author"}
    tables = re.findall(r"<table.*?>.*?</table>", text, flags=re.DOTALL | re.IGNORECASE)
    for table in tables:
        cells = re.findall(r"<td.*?>\s*(.*?)\s*</td>", table, flags=re.DOTALL | re.IGNORECASE)
        cleaned = [re.sub(r"<.*?>", "", c).strip() for c in cells]
        has_data = any(c and not any(kw in c.lower() for kw in keywords) for c in cleaned)
        if not has_data:
            text = text.replace(table, "")
    return re.sub(r"\n{3,}", "\n\n", text).strip()

def process_pdf_scan(pdf_path: str, output_dir: str) -> str:
    pages = convert_from_path(pdf_path, dpi=300)
    os.makedirs(output_dir, exist_ok=True)
    output = []
    for i, page in enumerate(pages):
        img_path = Path(output_dir) / f"temp_{i}.png"
        try:
            page.save(img_path)
            auto_rotate(img_path)
            res = model.infer(
                tokenizer,
                prompt=PDF_OCR_STYLE["prompt"],
                image_file=str(img_path),
                output_path=str(output_dir),
                base_size=PDF_OCR_STYLE["base_size"],
                image_size=PDF_OCR_STYLE["image_size"],
                crop_mode=PDF_OCR_STYLE["crop_mode"],
                save_results=PDF_OCR_STYLE["save_results"],
                test_compress=PDF_OCR_STYLE["test_compress"],
            )
            text = res["text"] if isinstance(res, dict) else str(res)
            output.append(f"\n## Page {i + 1}\n{clean_empty_tables(text)}")
        finally:
            img_path.unlink(missing_ok=True)
    return "\n".join(output)

def convert_document(file_path: str, output_dir: str = "/tmp") -> str:
    """Hàm tổng điều phối việc chuyển đổi file"""
    file_path_str = str(file_path)
    ext = Path(file_path_str).suffix.lower()
    
    if ext == ".doc":
        # logic _doc_to_docx có thể gom vào nếu cần, tạm thời pass để code gọn
        raise ValueError("Chưa hỗ trợ .doc trực tiếp, vui lòng dùng .docx hoặc PDF")

    if ext == ".pdf":
        if is_pdf_scan(file_path_str):
            logger.info("Phát hiện PDF Scan, đang kích hoạt DeepSeek-OCR...")
            content = process_pdf_scan(file_path_str, output_dir)
        else:
            logger.info("Phát hiện PDF Text, đang chạy Docling...")
            content = docling_converter.convert(file_path_str).document.export_to_markdown()
    elif ext == ".docx":
        doc = DocxDocument(file_path_str)
        content = "\n".join(p.text for p in doc.paragraphs if p.text.strip())
    elif ext in {".xlsx", ".xls"}:
        wb = openpyxl.load_workbook(file_path_str)
        content = []
        for sheet in wb.sheetnames:
            ws = wb[sheet]
            content.append(f"\n## Sheet: {sheet}")
            for row in ws.iter_rows(values_only=True):
                content.append(" | ".join(str(c) if c else "" for c in row))
        content = "\n".join(content)
    else:
        # Fallback to docling
        content = docling_converter.convert(file_path_str).document.export_to_markdown()

    if not content:
        raise ValueError("Nội dung trống sau khi chuyển đổi")
        
    return content

In [11]:
md_output_path = Path(pdf_file_path).with_suffix(".md")

try:
    print(f"🔄 Đang xử lý file: {pdf_file_path}")
    raw_md_text = convert_document(pdf_file_path, output_dir="/tmp")
    
    # Lưu ra file MD
    md_output_path.write_text(raw_md_text, encoding="utf-8")
    
    print(f"✅ Thành công! Đã lưu file Markdown tại: {md_output_path}")
    
    # In thử 500 ký tự đầu tiên để dev kiểm tra ngay trên Jupyter
    print("\n--- PREVIEW ---")
    print(raw_md_text[:500])
    
except Exception as e:
    traceback.print_exc()
    print(f"❌ Có lỗi xảy ra: {e}")

🔄 Đang xử lý file: /home/trung-ai/chatbot_hcns/doc/042025_AIPT_Đào tạo Hội nhập chung.docx
✅ Thành công! Đã lưu file Markdown tại: /home/trung-ai/chatbot_hcns/doc/042025_AIPT_Đào tạo Hội nhập chung.md

--- PREVIEW ---


ĐÀO TẠO HỘI NHẬP CHUNG
CHO NHÂN SỰ MỚI
Năm 2025
LỜI MỞ ĐẦU
Chào mừng bạn đã gia nhập vào đại gia đình Công ty Cổ phần AIPT Việt Nam. Chúng tôi rất vui mừng và trân trọng sự lựa chọn của bạn khi quyết định trở thành một phần của công ty chúng tôi. Quy trình đào tạo hội nhập này được thiết kế nhằm giúp bạn nhanh chóng hòa nhập vào môi trường làm việc mới, hiểu rõ về tầm nhìn, sứ mệnh, và giá trị cốt lõi của công ty. Chúng tôi mong muốn cung cấp cho bạn những thông tin cần thiết để bạn có thể bắt


# BƯỚC 2: CHUẨN HÓA DỮ LIỆU (CLEANING)
Tinh chỉnh bộ lọc Regex tại đây. Nếu có văn bản format lạ (như Phần, Mục, Điều không có dấu chấm...), hãy cập nhật thêm quy tắc vào hàm `clean_ocr_markdown`.

In [12]:
import re

def clean_ocr_markdown(text: str) -> str:
    """
    Hàm làm sạch rác OCR và khôi phục cấu trúc phân cấp Header.
    """
    # 1. Xóa các thẻ Page sinh ra do scan (VD: ## Page 5)
    text = re.sub(r'(?i)^#*\s*Page\s+\d+\s*$', '', text, flags=re.MULTILINE)
    
    # 2. Xóa các link hình ảnh rác 
    text = re.sub(r'!\[.*?\]\(.*?\)', '', text)
    
    # 3. Phục hồi thẻ Markdown cấp 1 (#) cho CHƯƠNG
    text = re.sub(r'^(CHƯƠNG\s+[IVXLCDM]+\b.*)$', r'# \1', text, flags=re.MULTILINE)
    
    # 4. Phục hồi thẻ Markdown cấp 2 (##) cho ĐIỀU
    text = re.sub(r'^(Điều\s+\d+\..*)$', r'## \1', text, flags=re.MULTILINE)
    
    # 5. Dọn dẹp bảng HTML rác do OCR bị lỗi bảng trống
    text = re.sub(r'<table.*?>.*?</table>', '', text, flags=re.DOTALL | re.IGNORECASE)
    
    # 6. Dọn dẹp khoảng trắng thừa
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    return text.strip()

In [13]:
from pathlib import Path
import os

# Đường dẫn file md thô từ BƯỚC 1
input_path_obj = Path(origin_input_file_path)

if not input_path_obj.exists():
    print(f"❌ Lỗi: Không tìm thấy file thô tại {origin_input_file_path}. Hãy chạy BƯỚC 1 trước.")
else:
    # Khởi tạo đường dẫn file đầu ra an toàn (_cleaned.md)
    output_file_path = input_path_obj.with_name(f"{input_path_obj.stem}_cleaned{input_path_obj.suffix}")

    print(f"⏳ Đang đọc dữ liệu thô từ: {input_path_obj.name}...")
    try:
        # Đọc dữ liệu gốc
        with open(origin_input_file_path, "r", encoding="utf-8") as f:
            raw_text = f.read()

        print("🧹 Đang chạy bộ lọc Regex chuẩn hóa Markdown...")
        cleaned_text = clean_ocr_markdown(raw_text)

        # Lưu dữ liệu đã làm sạch
        with open(output_file_path, "w", encoding="utf-8") as f:
            f.write(cleaned_text)

        print("-" * 50)
        print("✅ Đã làm sạch thành công!")
        print(f"📁 File sạch được lưu tại: {output_file_path}")
        print("-" * 50)
        
        # --- NGHIỆM THU TRỰC QUAN CHO DEV ---
        print("\n🔍 PREVIEW (Tìm thử thẻ # CHƯƠNG đầu tiên để kiểm tra):")
        preview_match = re.search(r'(# CHƯƠNG.*?)(?=\n#|\Z)', cleaned_text, flags=re.DOTALL)
        
        if preview_match:
            # In ra khoảng 500 ký tự của Chương tìm thấy
            print(preview_match.group(1)[:500] + "\n[...]")
        else:
            print("⚠️ Không tìm thấy từ khóa 'CHƯƠNG' nào được gắn thẻ #. Hãy kiểm tra lại Regex hoặc format văn bản gốc.")
            print(cleaned_text[:500] + "\n[...]")

    except Exception as e:
        print(f"❌ Đã xảy ra lỗi trong quá trình xử lý: {e}")

⏳ Đang đọc dữ liệu thô từ: 042025_AIPT_Đào tạo Hội nhập chung.md...
🧹 Đang chạy bộ lọc Regex chuẩn hóa Markdown...
--------------------------------------------------
✅ Đã làm sạch thành công!
📁 File sạch được lưu tại: /home/trung-ai/chatbot_hcns/doc/042025_AIPT_Đào tạo Hội nhập chung_cleaned.md
--------------------------------------------------

🔍 PREVIEW (Tìm thử thẻ # CHƯƠNG đầu tiên để kiểm tra):
⚠️ Không tìm thấy từ khóa 'CHƯƠNG' nào được gắn thẻ #. Hãy kiểm tra lại Regex hoặc format văn bản gốc.
ĐÀO TẠO HỘI NHẬP CHUNG
CHO NHÂN SỰ MỚI
Năm 2025
LỜI MỞ ĐẦU
Chào mừng bạn đã gia nhập vào đại gia đình Công ty Cổ phần AIPT Việt Nam. Chúng tôi rất vui mừng và trân trọng sự lựa chọn của bạn khi quyết định trở thành một phần của công ty chúng tôi. Quy trình đào tạo hội nhập này được thiết kế nhằm giúp bạn nhanh chóng hòa nhập vào môi trường làm việc mới, hiểu rõ về tầm nhìn, sứ mệnh, và giá trị cốt lõi của công ty. Chúng tôi mong muốn cung cấp cho bạn những thông tin cần thiết để bạn có thể

## BƯỚC 3: PHÂN MẢNH NGỮ CẢNH (CHUNKING) & LƯU TRỮ TRUNG GIAN
Sử dụng tokenizer của model Embedding để cắt nhỏ văn bản, bọc metadata và xuất ra file `.json` vào thư mục `staging_chunks` để lưu trữ và kiểm tra trước khi đẩy vào Vector DB.

In [14]:
import os
import json
import unicodedata
from transformers import AutoTokenizer
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

def normalize_vietnamese_text(text: str) -> str:
    """Chuẩn hóa Unicode tiếng Việt về chuẩn dựng sẵn NFC"""
    return unicodedata.normalize("NFC", text)

def process_markdown_for_rag_production(markdown_text: str, tokenizer):
    """Tiến hành phân mảnh văn bản dựa trên Tokenizer đã load"""
    # 0. Tiền xử lý văn bản
    clean_text = normalize_vietnamese_text(markdown_text)

    # 1. Tách theo Header cấu trúc
    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ]
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on,
        strip_headers=False
    )
    md_header_splits = markdown_splitter.split_text(clean_text)
    
    # 2. Sử dụng HuggingFace Tokenizer Splitter
    text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
        tokenizer=tokenizer,
        chunk_size=400,
        chunk_overlap=50,
        separators=[
            "\n\n", 
            "\n1. ", "\n2. ", "\n3. ", "\n4. ", "\n5. ", 
            "\na. ", "\nb. ", "\nc. ", "\nd. ", 
            "\n", ".", ";", " ", ""
        ]
    )
    
    # 3. Cắt nhỏ các chunk
    final_splits = text_splitter.split_documents(md_header_splits)
    
    # 4. Gắn Metadata / Breadcrumb (Context Enrichment)
    for doc in final_splits:
        metadata = doc.metadata
        path = " > ".join([v for k, v in metadata.items() if k.startswith("Header")])
        if path:
            doc.page_content = f"[{path}]\n{doc.page_content}"
            
    # Lọc bỏ các chunk quá ngắn (chỉ chứa tiêu đề hoặc rác)
    valid_chunks = [c for c in final_splits if len(c.page_content.strip()) > 100]        
    return valid_chunks

def export_chunks_to_json(chunks, output_dir: str, original_filename: str):
    """Lưu danh sách Document thành file JSON có cấu trúc đẹp để dev dễ đọc"""
    os.makedirs(output_dir, exist_ok=True)
    
    # Tạo tên file json dựa trên tên file gốc
    base_name = os.path.splitext(os.path.basename(original_filename))[0]
    json_filename = f"{base_name}_chunks.json"
    json_path = os.path.join(output_dir, json_filename)
    
    # Chuyển đổi list of Documents thành list of dicts
    data_to_save = [
        {"page_content": chunk.page_content, "metadata": chunk.metadata} 
        for chunk in chunks
    ]
    
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(data_to_save, f, ensure_ascii=False, indent=4)
        
    return json_path

In [15]:
# Cấu hình đường dẫn
model_id_or_path = "/home/trung-ai/chatbot_hcns/weights/embeddinggemma-300m" 
staging_dir = "/home/trung-ai/chatbot_hcns/staging_chunks"

if not os.path.exists(cleaned_text_file_path):
    print(f"❌ Không tìm thấy file nguồn: {cleaned_text_file_path}")
else:
    print(f"⏳ Đang load Tokenizer từ model local: {model_id_or_path}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id_or_path)
    
    print(f"⏳ Đang đọc nội dung file: {os.path.basename(cleaned_text_file_path)}...")
    with open(cleaned_text_file_path, "r", encoding="utf-8") as f:
        markdown_text = f.read()
        
    print("⚙️ Đang tiến hành phân mảnh...")
    try:
        chunks = process_markdown_for_rag_production(markdown_text, tokenizer)
        
        print("💾 Đang xuất dữ liệu ra định dạng JSON...")
        saved_json_path = export_chunks_to_json(chunks, staging_dir, cleaned_text_file_path)
        
        print("=" * 60)
        print(f"✅ HOÀN TẤT! Tổng số chunks hợp lệ: {len(chunks)}")
        print(f"📁 File JSON backup đã được lưu an toàn tại:\n   {saved_json_path}")
        print("=" * 60)
        
        # In thử nội dung chunk cuối cùng để nghiệm thu
        if chunks:
            print(f"\n🔍 PREVIEW CHUNK CUỐI (Kích thước: {len(chunks[-1].page_content)} ký tự):")
            print(chunks[-1].page_content)
            
    except Exception as e:
        print(f"❌ Lỗi trong quá trình chunking: {e}")

⏳ Đang load Tokenizer từ model local: /home/trung-ai/chatbot_hcns/weights/embeddinggemma-300m...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (3919 > 2048). Running this sequence through the model will result in indexing errors


⏳ Đang đọc nội dung file: 042025_AIPT_Đào tạo Hội nhập chung_cleaned.md...
⚙️ Đang tiến hành phân mảnh...
💾 Đang xuất dữ liệu ra định dạng JSON...
✅ HOÀN TẤT! Tổng số chunks hợp lệ: 14
📁 File JSON backup đã được lưu an toàn tại:
   /home/trung-ai/chatbot_hcns/staging_chunks/042025_AIPT_Đào tạo Hội nhập chung_cleaned_chunks.json

🔍 PREVIEW CHUNK CUỐI (Kích thước: 1314 ký tự):
Trách nhiệm bảo quản thông tin, hồ sơ, tài liệu, giây tờ thuộc trách nhiệm quản lý và công việc của mình.
Mọi nhân viên có người thân làm việc trong các công ty kinh doanh cùng lĩnh vực ngành hàng với Công ty phải báo cáo cho Phòng Nhân sự. Trường hợp nhân viên không tự giác khai báo cũng bị coi như đã có hành vi tiết lộ bí mật công nghệ, bí mật kinh doanh của Công ty.
CBNV không được thảo luận hoặc tiết lộ bất kỳ thông tin hoặc tải liệu nào có chứa thông tin mật với bất cứ người nào. Quy định vẫn áp dụng ngay cả đối với CBNV sau khi thôi làm việc cho Công ty vì bất cứ lý do gì, trừ các trường hợp sau:
Bảo mật lươn

In [ ]:
import json
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, SparseVectorParams
from qdrant_client import models
from langchain_core.documents import Document

# --- Cấu hình thông số ---
model_name = "/home/trung-ai/chatbot_hcns/weights/embeddinggemma-300m"
collection_name = "hcns_knowledge_base"

# 1. Khởi tạo mô hình Embedding
print(f"⏳ Đang tải mô hình Dense (Gemma-300m)...")
dense_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={'device': 'cuda', 'trust_remote_code': True}, 
    encode_kwargs={'normalize_embeddings': True} 
)

# Khởi tạo mô hình Sparse (BM25)
print("⏳ Đang tải mô hình Sparse (BM25)...")
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

print("🔌 Đang kết nối tới Qdrant (localhost:6333)...")
client = QdrantClient(url="http://localhost:6333")

# --- SỬA Ở ĐÂY: KIỂM TRA TRƯỚC KHI TẠO ---
if not client.collection_exists(collection_name=collection_name):
    print(f"🔨 Collection chưa tồn tại. Đang tạo mới Collection Hybrid: {collection_name}")
    client.create_collection(
        collection_name=collection_name,
        vectors_config={
            "dense-embedding": models.VectorParams(
                size=768, 
                distance=models.Distance.COSINE
            )
        },
        sparse_vectors_config={
            "sparse-embedding": models.SparseVectorParams(
                index=models.SparseIndexParams(on_disk=False)
            )
        }
    )
else:
    print(f"✅ Collection '{collection_name}' đã tồn tại. Tiến hành nạp thêm dữ liệu...")

# 3. Đọc dữ liệu từ file Staging JSON
print(f"📂 Đang đọc chunks từ: {json_input_path}")
with open(json_input_path, 'r', encoding='utf-8') as f:
    chunks_data = json.load(f)

# Chuyển đổi dữ liệu thô thành danh sách Document của LangChain
docs = [
    Document(page_content=item["page_content"], metadata=item["metadata"])
    for item in chunks_data
]

# 4. Thực thi Nhúng Hybrid và đẩy dữ liệu (Ingestion)
print(f"📥 Đang nhúng và nạp thêm {len(docs)} chunks vào Qdrant...")
qdrant = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=dense_embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="dense-embedding",
    sparse_vector_name="sparse-embedding"
)

# Hàm add_documents sẽ tự động nạp thêm (upsert) chứ không xóa dữ liệu cũ
doc_ids = qdrant.add_documents(documents=docs)

print("=" * 60)
print(f"🚀 Đã nạp thành công {len(doc_ids)} vectors vào Collection hiện tại.")
print("=" * 60)

INFO - Loading SentenceTransformer model from /home/trung-ai/chatbot_hcns/weights/embeddinggemma-300m.


⏳ Đang tải mô hình Dense (Gemma-300m)...


Loading weights: 100%|██████████| 314/314 [00:00<00:00, 12368.63it/s]
INFO - Loaded 14 prompts with these keys: ['query', 'document', 'BitextMining', 'Clustering', 'Classification', 'InstructionRetrieval', 'MultilabelClassification', 'PairClassification', 'Reranking', 'Retrieval', 'Retrieval-query', 'Retrieval-document', 'STS', 'Summarization']
INFO - HTTP Request: PUT http://localhost:6333/collections/hcns_knowledge_base "HTTP/1.1 409 Conflict"
INFO - HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"


⏳ Đang tải mô hình Sparse (BM25)...
🔌 Đang kết nối tới Qdrant (localhost:6333)...
🔨 Đang tạo mới Collection Hybrid: hcns_knowledge_base


UnexpectedResponse: Unexpected Response: 409 (Conflict)
Raw response content:
b'{"status":{"error":"Wrong input: Collection `hcns_knowledge_base` already exists!"},"time":0.000029516}'